### Basic Feature Engineering

In [ ]:
def drop_low_quality_columns(df, missing_threshold=0.5, drop_zero_std=True):
    """
    Evaluates and drops columns dynamically based on statistical quality.

    Parameters:
    df (pd.DataFrame): The input dataframe.
    missing_threshold (float): Drops columns where the percentage of NaNs/Infs exceeds this (e.g., 0.5 = 50%).
    drop_zero_std (bool): If True, drops columns where standard deviation is 0 (constant features).

    Returns:
    pd.DataFrame: Cleaned dataframe.
    """
    df_clean = df.copy()

    # 1. Convert Infs to NaNs first so they are counted together in the missing threshold
    df_clean.replace([np.inf, -np.inf], np.nan, inplace=True)

    # 2. Identify columns exceeding the NaN/Inf threshold
    # .isna().mean() calculates the exact percentage of missing values per column
    missing_pct = df_clean.isna().mean()
    high_nan_cols = missing_pct[missing_pct > missing_threshold].index.tolist()

    # 3. Identify columns with Standard Deviation of 0 (Constant Features)
    zero_std_cols = []
    if drop_zero_std:
        # Select only numerical columns to check standard deviation
        num_cols = df_clean.select_dtypes(include=[np.number]).columns
        # Find columns where std == 0
        zero_std_cols = num_cols[df_clean[num_cols].std() == 0].tolist()

    # 4. Combine the lists of bad columns and drop them all at once
    non_stationary = ['open', 'high', 'low', 'close', 'days', 'hours']
    #non_stationary = ['open', 'high', 'low', 'close']
    cols_to_drop = set(high_nan_cols + zero_std_cols + non_stationary)
    df_clean.drop(columns=list(cols_to_drop), inplace=True)

    # Optional: Print a report so you know exactly what was dynamically removed
    print(f"Dropped {len(cols_to_drop)} columns total.")
    print(f" -> {len(high_nan_cols)} due to >{missing_threshold*100}% missing data.")
    print(f" -> {len(zero_std_cols)} due to 0 standard deviation.")
    print(f" -> {len(non_stationary)} due to non_stationary.")

    return df_clean

In [ ]:
def cwts(target, target_col=None):
    if isinstance(target, pd.DataFrame):
        if target_col is None:
            raise ValueError("Please provide target_col when target is a DataFrame.")
        y = target[target_col].astype(int)
    elif isinstance(target, pd.Series):
        y = target.astype(int)
    else:
        y = np.asarray(target).astype(int)

    classes = np.bincount(y)
    if len(classes) < 2:
        raise ValueError("Need at least two classes for binary class weights.")
    if classes[0] == 0 or classes[1] == 0:
        raise ValueError("Both classes must be present in the target.")

    c0, c1 = classes[0], classes[1]
    w0 = (1 / c0) * (len(y) / 2)
    w1 = (1 / c1) * (len(y) / 2)
    return {0: w0, 1: w1}


### Scalling helper Functions

In [ ]:
import random
import tensorflow as tf

# 1. Build reproducible lightweight LSTM scorer for scaler search
def _set_reproducible_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    # Op-determinism is 2-5x slower on CPU and is not needed for model quality.
    # Seeds above already keep initialization reproducible.

def build_and_train_intermediate_lstm(
    X_tr_scaled, y_tr, X_te_scaled, y_te, class_weights,
    seqlen=21, hu=16, learning_rate=0.005, epochs=2, seed=42
 ):
    # Reset RNG and graph state for repeatable model initialization and training order.
    _set_reproducible_seed(seed)
    tf.keras.backend.clear_session()

    y_tr_values = y_tr.values if hasattr(y_tr, 'values') else y_tr
    y_te_values = y_te.values if hasattr(y_te, 'values') else y_te

    # Data Sequence Generation
    train_gen = TimeseriesGenerator(X_tr_scaled, y_tr_values, length=seqlen, batch_size=256)
    test_gen = TimeseriesGenerator(X_te_scaled, y_te_values, length=seqlen, batch_size=256)
    numfeat = X_tr_scaled.shape[1]

    # Lightweight model
    model = Sequential([
        Input(shape=(seqlen, numfeat), name='Input_Layer'),
        LSTM(units=hu, activation='elu', return_sequences=False, name='LSTM_Light'),
        Dense(units=1, activation='sigmoid', name='Output')
    ])

    opt = Adam(learning_rate=learning_rate, epsilon=1e-08)
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])

    # Keep shuffle disabled for stable, chronological financial time series training.
    model.fit(
        train_gen,
        epochs=epochs,
        validation_data=test_gen,
        class_weight=class_weights,
        shuffle=False,
        verbose=0
    )

    val_loss, val_accuracy = model.evaluate(test_gen, verbose=0)
    return val_accuracy, val_loss

def apply_scalers(scaler_dict, X_tr, X_te):
    groups = {'MinMaxScaler': [], 'RobustScaler': [], 'StandardScaler': []}
    for col, scaler_name in scaler_dict.items():
        if scaler_name not in groups:
            raise ValueError(f"Unsupported scaler: {scaler_name}")
        groups[scaler_name].append(col)

    transformers = []
    if groups['MinMaxScaler']:
        transformers.append(('minmax', MinMaxScaler(), groups['MinMaxScaler']))
    if groups['RobustScaler']:
        transformers.append(('robust', RobustScaler(), groups['RobustScaler']))
    if groups['StandardScaler']:
        transformers.append(('standard', StandardScaler(), groups['StandardScaler']))

    ct = ColumnTransformer(transformers=transformers, remainder='drop')

    return ct.fit_transform(X_tr), ct.transform(X_te)

def compare_recurrent_cells(
    X_tr_scaled, y_tr, X_te_scaled, y_te, class_weights,
    seqlen=21, hu=16, learning_rate=0.005, epochs=3, seed=42
):
    """
    Cheap proxy comparison of SimpleRNN vs GRU vs LSTM at fixed small capacity.
    Used as the initial-experimentation trail before fixing the architecture.
    """
    from tensorflow.keras.layers import SimpleRNN, GRU

    y_tr_values = y_tr.values if hasattr(y_tr, "values") else y_tr
    y_te_values = y_te.values if hasattr(y_te, "values") else y_te

    cell_types = {"SimpleRNN": SimpleRNN, "GRU": GRU, "LSTM": LSTM}
    results = {}

    print("=== Initial Experimentation: Recurrent Cell Comparison ===")
    for name, CellClass in cell_types.items():
        _set_reproducible_seed(seed)
        tf.keras.backend.clear_session()

        train_gen = TimeseriesGenerator(
            X_tr_scaled, y_tr_values, length=seqlen, batch_size=256
        )
        test_gen = TimeseriesGenerator(
            X_te_scaled, y_te_values, length=seqlen, batch_size=256
        )
        numfeat = X_tr_scaled.shape[1]

        model = Sequential([
            Input(shape=(seqlen, numfeat), name="Input_Layer"),
            CellClass(units=hu, activation="elu", return_sequences=False, name=f"{name}_Light"),
            Dense(units=1, activation="sigmoid", name="Output"),
        ])
        model.compile(
            optimizer=Adam(learning_rate=learning_rate, epsilon=1e-08),
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )
        model.fit(
            train_gen,
            epochs=epochs,
            validation_data=test_gen,
            class_weight=class_weights,
            shuffle=False,
            verbose=0,
        )
        val_loss, val_acc = model.evaluate(test_gen, verbose=0)
        results[name] = {"val_loss": float(val_loss), "val_accuracy": float(val_acc)}
        print(f"{name:>9}: val_loss={val_loss:.4f}  val_accuracy={val_acc:.4f}")

    best = min(results.items(), key=lambda kv: kv[1]["val_loss"])
    print(f"\nProxy winner (lowest val_loss): {best[0]}")
    return results


### Plotting Function for model performance of best model

In [9]:
def plot_model_performance(training_result, train_data, val_data, run_name="Model"):
    """
    Adaptively extracts the model and plots performance metrics in a 1x3 grid.
    Works for KerasTuner objects, standard training tuples, or raw Keras Models.
    """
    from sklearn.metrics import auc, roc_curve
    print(f"Extracting model and generating metrics for: {run_name}...")

    # 1. Adaptively extract the model, covering three scenario
    # Automated Hyperparameter Tuner(e.g KerasTuner), custom function returning (model, history) & standard Keras model training
    if hasattr(training_result, 'get_best_models'):
        best_model = training_result.get_best_models(num_models=1)[0]
    elif isinstance(training_result, tuple):
        best_model = training_result[0]
    else:
        best_model = training_result

    # 2. Extract true labels and generate predictions for VALIDATION (Test) data
    y_true_val = np.concatenate([y for x, y in val_data])
    y_pred_prob_val = best_model.predict(val_data, verbose=0)
    y_pred_val = (y_pred_prob_val > 0.5).astype(int)

    # 3. Extract true labels and generate predictions for TRAINING data
    y_true_train = np.concatenate([y for x, y in train_data])
    y_pred_prob_train = best_model.predict(train_data, verbose=0)

    # 4. Setup the 1x3 Matplotlib Figure
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle(f'{run_name} - Performance Metrics', fontsize=18, weight='bold', y=1.05)

    # ---------------------------------------------------------
    # Plot 1: Classification Report (Based on Validation Data)
    # ---------------------------------------------------------
    axes[0].axis('off')
    axes[0].set_title('Classification Report (Test Data)', fontsize=14, weight='bold', pad=15)
    report_str = classification_report(y_true_val, y_pred_val)
    axes[0].text(0.5, 0.5, report_str, {'fontsize': 13, 'fontfamily': 'monospace'}, ha='center', va='center',
                 transform=axes[0].transAxes, bbox=dict(boxstyle="round,pad=1", fc="#f8f9fa", ec="gray", alpha=0.9))

    # ---------------------------------------------------------
    # Plot 2: Confusion Matrix (Based on Validation Data)
    # ---------------------------------------------------------
    cm = confusion_matrix(y_true_val, y_pred_val)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1], cbar=False, annot_kws={"size": 16})
    axes[1].set_title('Confusion Matrix (Test Data)', fontsize=14, weight='bold', pad=15)
    axes[1].set_xlabel('Predicted Label', fontsize=12)
    axes[1].set_ylabel('True Label', fontsize=12)

    # ---------------------------------------------------------
    # Plot 3: ROC-AUC Curve (Training vs. Validation)
    # ---------------------------------------------------------
    # Calculate Training ROC
    fpr_train, tpr_train, _ = roc_curve(y_true_train, y_pred_prob_train)
    roc_auc_train = auc(fpr_train, tpr_train)

    # Calculate Validation ROC
    fpr_val, tpr_val, _ = roc_curve(y_true_val, y_pred_prob_val)
    roc_auc_val = auc(fpr_val, tpr_val)

    # Plot both curves
    axes[2].plot(fpr_train, tpr_train, color='royalblue', lw=2, label=f'Train ROC (AUC = {roc_auc_train:.3f})')
    axes[2].plot(fpr_val, tpr_val, color='darkorange', lw=2, label=f'Test ROC (AUC = {roc_auc_val:.3f})')

    axes[2].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    axes[2].set_xlim([0.0, 1.0])
    axes[2].set_ylim([0.0, 1.05])
    axes[2].set_xlabel('False Positive Rate', fontsize=12)
    axes[2].set_ylabel('True Positive Rate', fontsize=12)
    axes[2].set_title('ROC Curve (Train vs Test)', fontsize=14, weight='bold', pad=15)
    axes[2].legend(loc="lower right", fontsize=12)

    # Final Display Formatting
    plt.tight_layout()
    plt.show()


#################################################################################
########################Save and Evaluate Function###############################
#################################################################################


def save_and_evaluate(training_result, train_data, val_data, save_path='best_lstm_baseline.keras', run_name="LSTM Model"):
    """
    Adaptively extracts, saves, and evaluates the model whether it came
    from standard training, KerasTuner, or directly as a model object.
    """
    # 1. Adaptively extract the model
    if hasattr(training_result, 'get_best_models'):
        best_model = training_result.get_best_models(num_models=1)[0]
        print("\n[Extracted the best model from KerasTuner]")
    elif isinstance(training_result, tuple):
        best_model = training_result[0]
        print("\n[Extracted the model from the training tuple]")
    else:
        best_model = training_result
        print("\n[Using the provided Keras Model directly]")

    # 2. Save and Summarize
    best_model.save(save_path)
    print(f"Model successfully saved to {save_path}!\n")
    best_model.summary()

    # 3. Evaluate dynamically
    print(f"\n--- {run_name}: Validation Performance ---")
    val_scores = best_model.evaluate(val_data, verbose=0)
    for name, value in zip(best_model.metrics_names, val_scores):
        print(f" - {name}: {value:.4f}")

    print(f"\n--- {run_name}: Training Performance ---")
    train_scores = best_model.evaluate(train_data, verbose=0)
    for name, value in zip(best_model.metrics_names, train_scores):
        print(f" - {name}: {value:.4f}")

    return best_model

### Plotting function for ML models

In [ ]:
def plot_ml_model_performance(model, X_train, y_train, X_test, y_test, model_name="ML Model"):
    """
    Evaluates a scikit-learn compatible model and generates a 3-panel performance plot
    including a Classification Report, Confusion Matrix, and Train vs Test ROC Curve.
    """
    from sklearn.metrics import auc, roc_curve

    # 1. Generate Predictions
    y_pred_test = model.predict(X_test)

    # For ROC curve, we need probability scores for the positive class (class 1)
    # Check if the model has predict_proba (most classifiers do)
    if hasattr(model, "predict_proba"):
        y_prob_train = model.predict_proba(X_train)[:, 1]
        y_prob_test = model.predict_proba(X_test)[:, 1]
    else:
        # Fallback for models like linear SVM that use decision_function instead
        y_prob_train = model.decision_function(X_train)
        y_prob_test = model.decision_function(X_test)

    # 2. Set up the matplotlib figure (1 row, 3 columns)
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    fig.suptitle(f'{model_name} - Performance Metrics', fontsize=20, fontweight='bold', y=1.05)
    sns.set_theme(style="darkgrid") # Matches the seaborn style in your image

    # ==========================================
    # Subplot 1: Classification Report
    # ==========================================
    # Generate report string
    report_str = classification_report(y_test, y_pred_test, digits=2)

    axes[0].axis('off')
    axes[0].set_title('Classification Report (Test Data)', fontsize=14, fontweight='bold', pad=20)

    # Render text in a bounding box to match the image style
    axes[0].text(0.5, 0.5, report_str,
                 fontsize=16, fontfamily='monospace',
                 ha='center', va='center',
                 bbox=dict(boxstyle='round,pad=1', facecolor='#F8F9FA', edgecolor='gray', alpha=0.9))

    # ==========================================
    # Subplot 2: Confusion Matrix
    # ==========================================
    cm = confusion_matrix(y_test, y_pred_test)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1], cbar=False, annot_kws={"size": 20})
    axes[1].set_title('Confusion Matrix (Test Data)', fontsize=14, fontweight='bold', pad=20)
    axes[1].set_xlabel('Predicted Label', fontsize=14)
    axes[1].set_ylabel('True Label', fontsize=14)

    # Adjust tick labels for clarity
    axes[1].set_xticklabels(axes[1].get_xticklabels(), fontsize=14)
    axes[1].set_yticklabels(axes[1].get_yticklabels(), fontsize=14)

    # ==========================================
    # Subplot 3: ROC Curve (Train vs Test)
    # ==========================================
    # Calculate ROC metrics
    fpr_train, tpr_train, _ = roc_curve(y_train, y_prob_train)
    roc_auc_train = auc(fpr_train, tpr_train)

    fpr_test, tpr_test, _ = roc_curve(y_test, y_prob_test)
    roc_auc_test = auc(fpr_test, tpr_test)

    # Plot curves
    axes[2].plot(fpr_train, tpr_train, label=f'Train ROC (AUC = {roc_auc_train:.3f})', color='royalblue', lw=2)
    axes[2].plot(fpr_test, tpr_test, label=f'Test ROC (AUC = {roc_auc_test:.3f})', color='darkorange', lw=2)

    # Plot diagonal reference line
    axes[2].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')

    axes[2].set_xlim([0.0, 1.0])
    axes[2].set_ylim([0.0, 1.05])
    axes[2].set_xlabel('False Positive Rate', fontsize=12)
    axes[2].set_ylabel('True Positive Rate', fontsize=12)
    axes[2].set_title('ROC Curve (Train vs Test)', fontsize=14, fontweight='bold', pad=20)
    axes[2].legend(loc="lower right", fontsize=12)

    # Adjust layout to prevent overlap
    plt.tight_layout()
    plt.show()

## Setting up the Sequential LSTM Architecture

### Tensorflow settings

In [ ]:
import os
# ==========================================
# 1. Global Setup & TensorBoard Management
# ==========================================
MASTER_LOG_DIR = os.path.join('.', 'quant_model_logs')
os.makedirs(MASTER_LOG_DIR, exist_ok=True)

# Dictionary to track active TB sessions: { 'log_dir': 'url' }
_ACTIVE_TB_SESSIONS = {}

def launch_tensorboard_for_dir_for_colab(log_dir, start_port=6006):
    """Launches a dedicated TensorBoard for a specific directory to keep logs isolated."""
    global _ACTIVE_TB_SESSIONS

    # If a TensorBoard is already running for this exact directory, reuse it
    if log_dir in _ACTIVE_TB_SESSIONS:
        url = _ACTIVE_TB_SESSIONS[log_dir]
        print(f"\n[TensorBoard] Already active for this project at {url}.")
        print("[TensorBoard] Hit 'Refresh' (F5) in your browser.\n")
        return url

    tb = program.TensorBoard()
    port = start_port + len(_ACTIVE_TB_SESSIONS) # Increment port for new sessions

    try:
        tb.configure(argv=[None, '--logdir', log_dir, '--port', str(port)])
        url = tb.launch()
    except Exception:
        print(f"\n[TensorBoard] Port {port} busy. Finding a free port automatically...")
        tb.configure(argv=[None, '--logdir', log_dir])
        url = tb.launch()

    _ACTIVE_TB_SESSIONS[log_dir] = url
    print(f"\n[TensorBoard] Successfully launched dedicated session at: {url}\n")
    #webbrowser.open_new_tab(url)

    # --- COLAB specific code ---
    # 1. Parse the URL to find out which port TensorBoard actually used
    actual_port = urllib.parse.urlparse(url).port

    # 2. Ask Colab to open a new window for this port
    output.serve_kernel_port_as_window(actual_port)

    # 3. Generate a clickable proxy link as a backup in case the browser blocks the pop-up
    proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({actual_port})")
    print(f"[Colab Proxy] Click here to open TensorBoard: {proxy_url}\n")
    return url

In [ ]:
import os
import gc
import urllib.parse
from pathlib import Path
from tensorboard import program

def _resolve_master_log_dir():
    """
    KerasTuner + TensorBoard write thousands of small files. If the project
    lives on OneDrive, each write is synced and file-locked, so every extra
    run gets slower. Keep logs on a local disk in that case.
    """
    project_logs = Path(".").resolve() / "quant_model_logs"
    cwd = str(Path(".").resolve()).lower()
    on_onedrive = "onedrive" in cwd
    if on_onedrive:
        local_root = Path(os.environ.get("LOCALAPPDATA") or os.environ.get("TEMP") or ".")
        log_dir = local_root / "cqf_quant_model_logs"
        log_dir.mkdir(parents=True, exist_ok=True)
        print(f"[Logs] Project is on OneDrive. Writing tuner/TB logs locally to:\n       {log_dir}")
        return str(log_dir)
    project_logs.mkdir(parents=True, exist_ok=True)
    return str(project_logs)


MASTER_LOG_DIR = _resolve_master_log_dir()
os.makedirs(MASTER_LOG_DIR, exist_ok=True)

# One TensorBoard server per experiment family (baseline / random_search /
# bayesian_optimization / hyperband). Re-runs create new sub-folders inside the
# same family, so servers do not accumulate across a session.
_ACTIVE_TB_SESSIONS = {}  # serve_dir -> {"url", "port", "title"}
FIT_BATCH_SIZE = 128  # matches TimeseriesGenerator's default batch size used in the OLD notebook

TB_FAMILY_TITLES = {
    "baseline": "Baseline LSTM",
    "random_search": "Random Search",
    "bayesian_optimization": "Bayesian Optimization",
    "hyperband": "Hyperband",
}


def tb_family_dir(family):
    """Folder grouping every run of one experiment family."""
    path = os.path.join(MASTER_LOG_DIR, family)
    os.makedirs(path, exist_ok=True)
    return path


def _pretty_family(name):
    return TB_FAMILY_TITLES.get(name, name.replace("_", " ").title())


def _resolve_tb_url(launched_url):
    """Map TensorBoard launch URL to a usable browser URL (Codespaces or local)."""
    actual_port = urllib.parse.urlparse(launched_url).port
    codespace_name = os.environ.get("CODESPACE_NAME")
    domain = os.environ.get("GITHUB_CODESPACES_PORT_FORWARDING_DOMAIN", "app.github.dev")
    if codespace_name and actual_port:
        return f"https://{codespace_name}-{actual_port}.{domain}", actual_port
    if actual_port:
        return f"http://localhost:{actual_port}", actual_port
    return launched_url, actual_port


def _save_tb_url(url, log_dir, title=None):
    try:
        Path(log_dir).mkdir(parents=True, exist_ok=True)
        header = title or Path(log_dir).name
        (Path(log_dir) / "tensorboard_url.txt").write_text(f"{header}\n{url}\n", encoding="utf-8")
    except Exception:
        pass


def show_tensorboard_links():
    """One line per active board, labelled by experiment family."""
    if not _ACTIVE_TB_SESSIONS:
        print("[TensorBoard] No active sessions in this kernel.")
        print("[TensorBoard] Call restore_tensorboard_sessions() to relaunch from saved logs.")
        return
    for serve_dir, meta in _ACTIVE_TB_SESSIONS.items():
        print(f"[TensorBoard] {meta['title']:<22} {meta['url']}   (logdir: {serve_dir})")


def make_tb_callback(log_dir):
    """Scalars only. Skipping write_graph avoids large event files and OneDrive I/O."""
    os.makedirs(log_dir, exist_ok=True)
    return TensorBoard(
        log_dir=log_dir,
        write_graph=False,
        write_images=False,
        histogram_freq=0,
        update_freq="epoch",
        profile_batch=0,
    )


def make_windowed_arrays(X, y, seq_len):
    """Same indexing as Keras TimeseriesGenerator (window X[i:i+L], target y[i+L])."""
    from numpy.lib.stride_tricks import sliding_window_view
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y).reshape(-1)
    seq_len = int(seq_len)
    n = len(X) - seq_len
    if n <= 0:
        raise ValueError("Not enough rows for the requested seq_len.")
    Xw = sliding_window_view(X, seq_len, axis=0)[:n]
    # sliding_window_view on 2D data is (n, features, seq_len); LSTM needs (n, seq_len, features)
    Xw = np.transpose(Xw, (0, 2, 1))
    yw = y[seq_len:]
    return Xw, yw


def make_tf_dataset(X, y, seq_len, batch_size=FIT_BATCH_SIZE, shuffle=False):
    """Pre-windowed tf.data pipeline. Faster than TimeseriesGenerator on CPU."""
    Xw, yw = make_windowed_arrays(X, y, seq_len)
    ds = tf.data.Dataset.from_tensor_slices((Xw, yw))
    if shuffle:
        ds = ds.shuffle(len(Xw), reshuffle_each_iteration=False)
    return ds.batch(int(batch_size)).prefetch(tf.data.AUTOTUNE)


def _tb_serve_target(log_dir):
    """
    Resolve (serve_dir, window_title, run_name) for a run directory.

    A run at <MASTER>/<family>/<run> is served by a single board mounted on
    <family> and titled after that family, so the browser tab identifies which
    tuner produced it. A legacy run sitting directly under <MASTER> is served
    on its own.
    """
    run_path = Path(log_dir).resolve()
    master = Path(MASTER_LOG_DIR).resolve()
    parent = run_path.parent
    if parent == master:
        return str(run_path), _pretty_family(run_path.name), run_path.name
    return str(parent), _pretty_family(parent.name), run_path.name


def _run_scoped_url(base_url, run_name):
    """Deep link that filters the board down to one run."""
    return f"{base_url}/#scalars&runFilter={urllib.parse.quote(run_name)}"


def launch_tensorboard_for_dir(log_dir, start_port=6006, title=None):
    """
    Start (or reuse) the TensorBoard for this run's experiment family and return
    a link scoped to this run.

    The window title names the family (Random Search / Bayesian Optimization /
    Hyperband / Baseline LSTM) so concurrent boards stay distinguishable, and the
    returned URL pre-filters the runs list to `log_dir`.
    """
    global _ACTIVE_TB_SESSIONS
    os.makedirs(log_dir, exist_ok=True)
    serve_dir, family_title, run_name = _tb_serve_target(log_dir)
    window_title = title or family_title

    meta = _ACTIVE_TB_SESSIONS.get(serve_dir)
    if meta is None:
        tb = program.TensorBoard()
        port = start_port + len(_ACTIVE_TB_SESSIONS)
        argv = [
            None, "--logdir", serve_dir, "--port", str(port),
            "--reload_interval", "15", "--window_title", window_title,
        ]
        try:
            tb.configure(argv=argv)
            launched_url = tb.launch()
        except Exception:
            tb.configure(argv=[None, "--logdir", serve_dir, "--window_title", window_title])
            launched_url = tb.launch()

        base_url, actual_port = _resolve_tb_url(launched_url)
        meta = {"url": base_url, "port": actual_port, "title": window_title}
        _ACTIVE_TB_SESSIONS[serve_dir] = meta

    run_url = _run_scoped_url(meta["url"], run_name)
    _save_tb_url(run_url, log_dir, f"{meta['title']} | {run_name}")
    print(f"[TensorBoard] {meta['title']:<22} run={run_name} -> {run_url}")
    return run_url


def _has_tb_events(path):
    """True if path (or any descendant) contains TensorBoard event files."""
    p = Path(path)
    if not p.is_dir():
        return False
    return next(p.rglob("events.out.tfevents*"), None) is not None


def _latest_child_with_events(dir_path):
    """Newest immediate subdirectory that still has event files, else dir itself."""
    root = Path(dir_path)
    children = [c for c in root.iterdir() if c.is_dir() and _has_tb_events(c)]
    if children:
        return str(max(children, key=lambda c: c.stat().st_mtime))
    if _has_tb_events(root):
        return str(root)
    return None


def _discover_tb_restore_targets(master_dir=None):
    """
    Build (log_dir, title) pairs for boards that can be restored from disk.

    - Known family folders (e.g. baseline/) -> latest nested run
    - Flat legacy runs (rstrail_*, botrial_*, hbtrial_*) -> latest of each prefix
    """
    master = Path(master_dir or MASTER_LOG_DIR).resolve()
    if not master.is_dir():
        return []

    targets = []
    covered_prefixes = set()

    for family, title in TB_FAMILY_TITLES.items():
        fam_path = master / family
        latest = _latest_child_with_events(fam_path) if fam_path.is_dir() else None
        if latest:
            targets.append((latest, title))
            covered_prefixes.add(family)

    # Flat run folders written directly under MASTER_LOG_DIR by the tuner cells
    prefix_map = {
        "rstrail": "Random Search",
        "botrial": "Bayesian Optimization",
        "hbtrial": "Hyperband",
        "lstm_baseline": "Baseline LSTM",
    }
    latest_by_prefix = {}
    for child in master.iterdir():
        if not child.is_dir() or child.name in TB_FAMILY_TITLES:
            continue
        if not _has_tb_events(child):
            continue
        for prefix, title in prefix_map.items():
            if child.name.startswith(prefix + "_") or child.name == prefix:
                mtime = child.stat().st_mtime
                prev = latest_by_prefix.get(prefix)
                if prev is None or mtime > prev[0]:
                    latest_by_prefix[prefix] = (mtime, child, title)
                break

    # Skip lstm_baseline_* flats when the baseline/ family board already covers them
    if "baseline" in covered_prefixes:
        latest_by_prefix.pop("lstm_baseline", None)

    for prefix, (_, path, title) in sorted(latest_by_prefix.items()):
        targets.append((str(path), f"{title} | {path.name}"))

    return targets


def restore_tensorboard_sessions(start_port=6006, master_dir=None):
    """
    Re-launch TensorBoard on persisted event logs after Cursor/kernel restart.

    Logs on disk survive; only the HTTP servers need restarting. Safe to call
    repeatedly - launch_tensorboard_for_dir reuses boards already active in
    this kernel. New training runs keep writing into (and replacing) the same
    family / latest-run slots.
    """
    targets = _discover_tb_restore_targets(master_dir)
    if not targets:
        print(f"[TensorBoard] No event logs found under {master_dir or MASTER_LOG_DIR}")
        return {}

    print(f"[TensorBoard] Restoring {len(targets)} board(s) from disk...")
    restored = {}
    for log_dir, title in targets:
        try:
            url = launch_tensorboard_for_dir(log_dir, start_port=start_port, title=title)
            restored[log_dir] = url
        except Exception as exc:
            print(f"[TensorBoard] Failed to restore {title}: {exc}")
    return restored


### Starting with Base Model

In [ ]:
"""A standard model builder completely independent of KerasTuner."""
# Build a function accepting seqlen, numfeat, hu (default=64), learning_rate
def build_base_lstm(seqlen, numfeat, hu=64, learning_rate=0.001):
    tf.keras.backend.clear_session()
    model = Sequential()

    # Explicit Input Layer
    model.add(Input(shape=(seqlen, numfeat), name='Input_Layer'))

    model.add(LSTM(units=hu*2, activation='elu', return_sequences=True,
                   kernel_regularizer=l2(0.001), recurrent_regularizer=l2(0.001), name='LSTM1'))
    model.add(Dropout(0.4, name='Dropout1'))

    model.add(LSTM(units=hu, activation='elu', return_sequences=False,
                   kernel_regularizer=l2(0.001), recurrent_regularizer=l2(0.001), name='LSTM2'))
    model.add(Dropout(0.4, name='Dropout2'))

    model.add(Dense(units=1, activation='sigmoid', name='Output'))

    opt = Adam(learning_rate=learning_rate, epsilon=1e-08, weight_decay=0.0)
    model.compile(optimizer=opt, loss=BinaryCrossentropy(), metrics=['accuracy', Precision(), Recall()])

    return model


def run_standard_training(project_name, train_data, val_data, seqlen, numfeat,
                          hu=64, learning_rate=0.001, epochs=100, class_weight=None,
                          extra_callbacks=None, seed=42):
    """Executes a standard training run WITHOUT hyperparameter tuning."""
    project_log_dir = os.path.join(tb_family_dir('baseline'), project_name)
    os.makedirs(project_log_dir, exist_ok=True)

    # Force deterministic initialization/training for reproducible baselines.
    if '_set_reproducible_seed' in globals():
        _set_reproducible_seed(seed)
    else:
        random.seed(seed)
        np.random.seed(seed)
        tf.keras.utils.set_random_seed(seed)
        pass

    model = build_base_lstm(seqlen, numfeat, hu=hu, learning_rate=learning_rate)

    model.build(input_shape=(None, seqlen, numfeat))
    _ = model(tf.zeros((1, seqlen, numfeat)))  # Dummy pass to lock the graph

    tb_callback = make_tb_callback(project_log_dir)
    callbacks = [tb_callback] + (extra_callbacks if extra_callbacks else [])

    print(f"--- Starting Standard Run: {project_name} ---")
    launch_tensorboard_for_dir(project_log_dir)

    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=epochs,
        callbacks=callbacks,
        class_weight=class_weight,
        shuffle=False
    )

    gc.collect()
    return model, history


### Moving toward hyperparameter Tuning

In [ ]:
# Tunable 3-layer LSTM search space (joint architecture + hyperparameter search).
# Kept intentionally narrow (matches the original/OLD notebook): small unit ranges
# and no L2 term keep each trial cheap, since HPO already runs dozens of trials.
def build_model(hp):
    tf.keras.backend.clear_session()

    model = Sequential()
    # Explicit Input layer for perfect TensorBoard Graphs
    model.add(Input(shape=(SEQLEN, NUMFEAT), name='Input_Layer'))

    # Small capacity by design: keeps per-trial training cost low across dozens of trials.
    hp_units1 = hp.Int('units1', min_value=4, max_value=32, step=4)
    hp_units2 = hp.Int('units2', min_value=4, max_value=32, step=4)
    hp_units3 = hp.Int('units3', min_value=4, max_value=32, step=4)

    hp_dropout1 = hp.Float('Dropout_rate_1', min_value=0, max_value=0.5, step=0.1)
    hp_dropout2 = hp.Float('Dropout_rate_2', min_value=0, max_value=0.5, step=0.1)

    hp_activation1 = hp.Choice('activation_1', values=['relu', 'elu'], ordered=False)
    hp_activation2 = hp.Choice('activation_2', values=['relu', 'elu'], ordered=False)
    hp_activation3 = hp.Choice('activation_3', values=['relu', 'elu'], ordered=False)

    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.add(LSTM(hp_units1, activation=hp_activation1, return_sequences=True, name='LSTM1'))
    model.add(Dropout(hp_dropout1, name='Dropout1'))

    model.add(LSTM(hp_units2, activation=hp_activation2, return_sequences=True, name='LSTM2'))
    model.add(Dropout(hp_dropout2, name='Dropout2'))

    model.add(LSTM(hp_units3, activation=hp_activation3, return_sequences=False, name='LSTM3'))

    model.add(Dense(units=1, activation='sigmoid', name='Output'))

    opt = Adam(learning_rate=hp_learning_rate, epsilon=1e-08)

    model.compile(optimizer=opt, loss=BinaryCrossentropy(), metrics=['accuracy', Precision(), Recall()])

    return model

# def run_tuner_search(project_name, train_data, val_data, seqlen, numfeat,
#                      epochs=50, class_weight=None, extra_callbacks=None):
#     tf.keras.utils.set_random_seed(42)
#     """Executes KerasTuner hyperparameter search."""
#     project_log_dir = os.path.join(MASTER_LOG_DIR, project_name)

#     # Closure to bridge KerasTuner 'hp' with our standard model builder
#     def tuner_builder_wrapper(hp):
#         tuned_hu = hp.Int('hu', min_value=10, max_value=256, step=16)
#         tuned_lr = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
#         return build_base_lstm(seqlen, numfeat, hu=tuned_hu, learning_rate=tuned_lr)

#     tuner = kt.Hyperband(tuner_builder_wrapper, objective="val_accuracy", max_epochs=epochs, factor=2,
#                          directory=MASTER_LOG_DIR, project_name=project_name, overwrite=True)

#     #tb_callback = TensorBoard(log_dir=project_log_dir, write_graph=True, update_freq='epoch')
#     tb_callback = TensorBoard(log_dir=project_log_dir, write_graph=True, update_freq='epoch', profile_batch=0)
#     callbacks = [tb_callback] + (extra_callbacks if extra_callbacks else [])

#     print(f"--- Starting Tuner Search: {project_name} ---")
#     launch_tensorboard_for_dir(project_log_dir, title=project_name)

#     tuner.search(train_data, validation_data=val_data, epochs=epochs,
#                  callbacks=callbacks, class_weight=class_weight, shuffle=False)

#     return tuner


## Strategy testing

In [2]:
def plot_strategy_performance(df, ypred, start_date=None, end_date=None, show_plot=True):
    """
    Plots the performance of a Deep Learning strategy vs the Market.
    """
    import matplotlib.dates as mdates
    # 1. Align the original dataframe slice with your prediction length
    df_full = df.iloc[-len(ypred):].copy()
    df_full.index = pd.to_datetime(df_full.index)

    # 2. Calculate Returns and Signals on the FULL dataset first
    df_full['Market_Returns'] = df_full['close'].pct_change()
    df_full['DL_Signal'] = ypred.flatten()
    df_full['Executed_Signal'] = df_full['DL_Signal'].shift(1)
    df_full['Strategy_Returns'] = np.where(df_full['Executed_Signal'] == 1, df_full['Market_Returns'], 0)

    # 3. Apply Date Filters if provided
    if start_date:
        df_full = df_full[df_full.index >= pd.to_datetime(start_date)]
    if end_date:
        df_full = df_full[df_full.index <= pd.to_datetime(end_date)]

    # 4. Calculate Cumulative Growth
    df_full['Cumulative_Market'] = (1 + df_full['Market_Returns'].fillna(0)).cumprod()
    df_full['Cumulative_Strategy'] = (1 + df_full['Strategy_Returns'].fillna(0)).cumprod()

    # 5. Plotting Logic - ONLY executes if show_plot is True
    if show_plot:
        fig, ax1 = plt.subplots(figsize=(20, 10))
        plt.title('Performance: Market vs. Long-Only Deep Learning Strategy', size=18, pad=20)

        # --- Primary Y-Axis: Cumulative Returns ---
        ax1.plot(df_full.index, df_full['Cumulative_Market'], label='Market Returns', color='blue', linewidth=2)
        ax1.plot(df_full.index, df_full['Cumulative_Strategy'], label='Long-Only DL Strategy', color='orange', linewidth=2)
        ax1.set_ylabel('Cumulative Return (Base 1.0)', size=14)
        ax1.set_xlabel('Date', size=14)
        ax1.grid(True, linestyle='--', alpha=0.5)

        # Highlight Long positions
        ax1.fill_between(df_full.index,
                         ax1.get_ylim()[0], ax1.get_ylim()[1],
                         where=(df_full['Executed_Signal'] == 1),
                         color='green', alpha=0.15, label='Long Position Active')

        # --- Secondary Y-Axis: Close Price ---
        ax2 = ax1.twinx()
        ax2.plot(df_full.index, df_full['close'], label='Close Price', color='black', linewidth=1.5, alpha=0.4, linestyle='-.')
        ax2.set_ylabel('Close Price', size=14, color='black', alpha=0.7)

        # Combine Legends
        lines_1, labels_1 = ax1.get_legend_handles_labels()
        lines_2, labels_2 = ax2.get_legend_handles_labels()
        ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left', fontsize=12)

        # X-Axis Date Formatting
        ax1.xaxis.set_major_locator(mdates.MonthLocator())
        ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

        plt.tight_layout()
        plt.show()

    # Always return the dataframe so you can debug
    return df_full

In [ ]:
def get_period_metrics(daily_returns, rf_rate=0.0):
    """
    Calculates Return, Annualized Sharpe, and Max Drawdown for any given slice of daily returns.
    """
    # If the period has no data, return zeros
    if len(daily_returns) == 0:
        return pd.Series({'Return': 0.0, 'Sharpe': 0.0, 'Max_DD': 0.0})

    # 1. Total Period Return
    period_return = (1 + daily_returns).prod() - 1

    # 2. Annualized Sharpe Ratio for the Period
    std_dev = daily_returns.std()
    if std_dev > 0:
        # Annualized based on 252 trading days
        sharpe = (daily_returns.mean() * 252 - rf_rate) / (std_dev * np.sqrt(252))
    else:
        sharpe = 0.0

    # 3. Maximum Drawdown for the Period
    cum_ret = (1 + daily_returns).cumprod()
    rolling_max = cum_ret.cummax()
    drawdown = (cum_ret - rolling_max) / rolling_max
    max_dd = drawdown.min()

    return pd.Series({'Return': period_return, 'Sharpe': sharpe, 'Max_DD': max_dd})

def generate_modular_tables(df, ypred, tc_bps=5, slippage_bps=5):
    """
    Calculates net returns and applies a modular metric function across multiple timeframes.
    """
    # 1. Align Data & Calculate Net Returns (including friction)
    dfr = df.iloc[-len(ypred):].copy()
    dfr.index = pd.to_datetime(dfr.index)

    dfr['Market_Returns'] = dfr['close'].pct_change()
    dfr['Executed_Signal'] = ypred.flatten()
    dfr['Executed_Signal'] = dfr['Executed_Signal'].shift(1).fillna(0)

    # Friction Calculation
    dfr['Signal_Change'] = dfr['Executed_Signal'].diff().fillna(0).abs()
    dfr['Friction'] = dfr['Signal_Change'] * ((tc_bps + slippage_bps) / 10000)
    dfr['Net_Returns'] = np.where(dfr['Executed_Signal'] == 1, dfr['Market_Returns'], 0) - dfr['Friction']

    # 2. Modular Timeframe Grouping
    # You can add or remove any valid pandas offset aliases here
    timeframes = {
        'Yearly': 'YE',
        'Quarterly': 'QE',
        'Monthly': 'ME',
        'Weekly': 'W'
    }

    results = {}

    # Iterate through the dictionary, grouping and calculating simultaneously
    for name, freq in timeframes.items():
        # Apply the custom metric function to the grouped timeframe
        metrics_df = dfr.groupby(pd.Grouper(freq=freq))['Net_Returns'].apply(get_period_metrics).unstack()
        results[name] = metrics_df

        # Format for terminal printing
        fmt_df = metrics_df.copy()
        fmt_df['Return'] = fmt_df['Return'].apply(lambda x: f"{x:.2%}")
        fmt_df['Sharpe'] = fmt_df['Sharpe'].apply(lambda x: f"{x:.2f}")
        fmt_df['Max_DD'] = fmt_df['Max_DD'].apply(lambda x: f"{x:.2%}")

        print(f"\n{'='*45}\n 📅 {name.upper()} METRICS\n{'='*45}")

        # Only print the tail for weekly to avoid terminal flooding
        if name == 'Weekly':
            print(fmt_df.tail(10))
        else:
            print(fmt_df)

    return results

In [1]:
# import matplotlib.pyplot as plt
# import numpy as np
from matplotlib.ticker import FuncFormatter
def plot_period_metrics(metrics_df, timeframe_name="Metrics"):
    """
    Creates a crisp, 3-panel chart (Return, Sharpe, Max Drawdown) for a given timeframe.
    Pass in tables['Yearly'], tables['Monthly'], etc.
    """
    # 1. Clean Data & Prevent Clutter
    df = metrics_df.dropna().copy()

    # If the dataframe is huge (like 5 years of weekly data),
    # limit to the last 30 periods to keep the chart readable.
    if len(df) > 30:
        df = df.tail(30)
        timeframe_name += " (Last 30 Periods)"

    # Clean up the X-axis labels (strip timestamps if present)
    x_labels = df.index.astype(str).str[:10]
    x = np.arange(len(df))

    # 2. Setup the Figure
    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
    fig.suptitle(f'Strategy Performance: {timeframe_name}', fontsize=16, fontweight='bold', y=0.98)

    # --- Panel 1: Net Return ---
    # Green for positive, Red for negative
    colors_ret = ['seagreen' if val >= 0 else 'indianred' for val in df['Return']]
    axes[0].bar(x, df['Return'], color=colors_ret, alpha=0.85)
    axes[0].set_ylabel('Net Return', fontsize=12, fontweight='bold')
    axes[0].yaxis.set_major_formatter(FuncFormatter(lambda y, _: '{:.1%}'.format(y)))

    # --- Panel 2: Sharpe Ratio ---
    # Blue for positive alpha, Grey for negative
    colors_shp = ['steelblue' if val >= 0 else 'lightslategray' for val in df['Sharpe']]
    axes[1].bar(x, df['Sharpe'], color=colors_shp, alpha=0.85)
    axes[1].set_ylabel('Ann. Sharpe', fontsize=12, fontweight='bold')

    # --- Panel 3: Max Drawdown ---
    # Always red (risk)
    axes[2].bar(x, df['Max_DD'], color='firebrick', alpha=0.85)
    axes[2].set_ylabel('Max Drawdown', fontsize=12, fontweight='bold')
    axes[2].yaxis.set_major_formatter(FuncFormatter(lambda y, _: '{:.1%}'.format(y)))

    # 3. Clean Formatting for all panels
    for ax in axes:
        ax.axhline(0, color='black', linewidth=1) # Zero line
        ax.grid(axis='y', linestyle='--', alpha=0.4) # Light horizontal grid
        ax.spines['top'].set_visible(False) # Remove top border
        ax.spines['right'].set_visible(False) # Remove right border

    # X-axis formatting on the bottom plot only
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(x_labels, rotation=45, ha='right', fontsize=10)

    plt.tight_layout()
    plt.show()

### Final model visualization
Runs the full backtest visualization suite (equity curve, optional zoomed window, and
Yearly/Monthly/Weekly period charts) on whichever model was ultimately selected
(MCDM winner, backtest-composite winner, or any single candidate), instead of the
naive lowest-validation-loss model picked earlier in the notebook.

In [ ]:
def visualize_final_model_strategy(model, df, val_data, model_name="Final Selected Model",
                                    tc_bps=5, slippage_bps=5,
                                    zoom_start=None, zoom_end=None,
                                    show_yearly=True, show_quarterly=False,
                                    show_monthly=True, show_weekly=True):
    """
    Runs the full "Strategy Performance in real world" visualization suite on a
    single, already-selected model: full-period equity curve, an optional zoomed
    window, and Yearly/Monthly/Weekly period-metric charts.

    Predicts exactly once and reuses `ypred` across every chart/table below it
    (the original notebook cells re-derived signals per cell), so calling this on
    a large val_data set only pays the LSTM forward-pass cost a single time.
    """
    print(f"\n{'=' * 90}\nBACKTEST VISUALIZATION: {model_name}\n{'=' * 90}")

    y_prob = model.predict(val_data, verbose=0)
    ypred = (y_prob > 0.5).astype(int)

    # 1. Full-period equity curve (gross, long/cash) vs Buy & Hold.
    plot_strategy_performance(df, ypred)

    # 2. Optional zoomed-in window (e.g. a specific drawdown or regime).
    if zoom_start or zoom_end:
        plot_strategy_performance(df, ypred, start_date=zoom_start, end_date=zoom_end)

    # 3. Cost-aware period tables + charts (single computation, reused for all panels).
    tables = generate_modular_tables(df, ypred, tc_bps=tc_bps, slippage_bps=slippage_bps)
    period_flags = {
        "Yearly": show_yearly,
        "Quarterly": show_quarterly,
        "Monthly": show_monthly,
        "Weekly": show_weekly,
    }
    for period, show in period_flags.items():
        if show and period in tables:
            plot_period_metrics(tables[period], timeframe_name=period)

    return {"ypred": ypred, "tables": tables}

### Multi-model backtest comparison
- Backtests every candidate on the same lagged, cost-aware long/cash rule
- Reports classification quality and trading P&L side by side
- Ranks models on trading metrics and selects the backtest winner


In [ ]:
def _extract_keras_model(obj):
    """Return a Keras model from a tuner, (model, history) tuple, or model."""
    if hasattr(obj, "get_best_models"):
        return obj.get_best_models(num_models=1)[0]
    if isinstance(obj, tuple):
        return obj[0]
    return obj


def _align_cost_aware_backtest(df, ypred, tc_bps=5, slippage_bps=5):
    """
    Align predictions to the last len(ypred) bars and apply the same causal rule
    used elsewhere: execute previous-bar signal, deduct round-trip friction.
    """
    ypred = np.asarray(ypred).reshape(-1)
    dfr = df.iloc[-len(ypred):].copy()
    dfr.index = pd.to_datetime(dfr.index)
    dfr["Market_Returns"] = dfr["close"].pct_change()
    dfr["DL_Signal"] = ypred
    dfr["Executed_Signal"] = dfr["DL_Signal"].shift(1).fillna(0)
    dfr["Signal_Change"] = dfr["Executed_Signal"].diff().fillna(0).abs()
    dfr["Friction"] = dfr["Signal_Change"] * ((tc_bps + slippage_bps) / 10000.0)
    mkt = dfr["Market_Returns"].fillna(0.0)
    dfr["Gross_Returns"] = np.where(dfr["Executed_Signal"] == 1, mkt, 0.0)
    dfr["Net_Returns"] = dfr["Gross_Returns"] - dfr["Friction"]
    dfr["Cumulative_Market"] = (1.0 + mkt).cumprod()
    dfr["Cumulative_Strategy"] = (1.0 + dfr["Net_Returns"]).cumprod()
    return dfr


def _round_trip_stats(dfr):
    """P&L statistics on consecutive long episodes (round trips)."""
    sig = dfr["Executed_Signal"].astype(int).to_numpy()
    rets = dfr["Net_Returns"].to_numpy()
    trades, acc, hold, in_trade = [], 1.0, 0, False
    for s, r in zip(sig, rets):
        if s == 1:
            in_trade = True
            acc *= (1.0 + r)
            hold += 1
        elif in_trade:
            trades.append((acc - 1.0, hold))
            in_trade, acc, hold = False, 1.0, 0
    if in_trade:
        trades.append((acc - 1.0, hold))
    if not trades:
        return {
            "Round Trips": 0,
            "Trade Win Rate": np.nan,
            "Avg Trade": np.nan,
            "Avg Hold (bars)": np.nan,
            "Payoff Ratio": np.nan,
            "Best Trade": np.nan,
            "Worst Trade": np.nan,
        }
    pnl = np.array([t[0] for t in trades], dtype=float)
    holds = np.array([t[1] for t in trades], dtype=float)
    wins, losses = pnl[pnl > 0], pnl[pnl < 0]
    payoff = (wins.mean() / abs(losses.mean())) if len(wins) and len(losses) else np.nan
    return {
        "Round Trips": int(len(pnl)),
        "Trade Win Rate": float((pnl > 0).mean()),
        "Avg Trade": float(pnl.mean()),
        "Avg Hold (bars)": float(holds.mean()),
        "Payoff Ratio": float(payoff) if payoff == payoff else np.nan,
        "Best Trade": float(pnl.max()),
        "Worst Trade": float(pnl.min()),
    }


def _max_dd_duration(drawdown):
    dur = max_dur = 0
    for x in drawdown:
        if x < 0:
            dur += 1
            max_dur = max(max_dur, dur)
        else:
            dur = 0
    return max_dur


def _compute_strategy_metrics(dfr, rf_rate=0.0):
    r = dfr["Net_Returns"]
    g = dfr["Gross_Returns"]
    m = dfr["Market_Returns"].fillna(0.0)
    sig = dfr["Executed_Signal"]

    bars_per_day = dfr.groupby(dfr.index.normalize()).size().median()
    bars_per_day = 1.0 if pd.isna(bars_per_day) or bars_per_day <= 0 else float(bars_per_day)
    ppy = bars_per_day * 252.0
    years = max((dfr.index[-1] - dfr.index[0]).days / 365.25, 1e-9)

    total_net = float((1.0 + r).prod() - 1.0)
    total_gross = float((1.0 + g).prod() - 1.0)
    total_mkt = float((1.0 + m).prod() - 1.0)
    cagr = float((1.0 + total_net) ** (1.0 / years) - 1.0) if total_net > -1 else -1.0
    mkt_cagr = float((1.0 + total_mkt) ** (1.0 / years) - 1.0) if total_mkt > -1 else -1.0

    r_std = float(r.std(ddof=1)) if len(r) > 1 else 0.0
    vol = r_std * np.sqrt(ppy) if r_std > 0 else 0.0
    sharpe = ((float(r.mean()) * ppy) - rf_rate) / (r_std * np.sqrt(ppy)) if r_std > 0 else 0.0
    # Sortino: full-sample downside deviation of returns below 0 (not std of negatives only)
    r_arr = r.to_numpy(dtype=float)
    dstd = float(np.sqrt(np.mean(np.minimum(r_arr, 0.0) ** 2))) if len(r_arr) else 0.0
    sortino = ((float(r.mean()) * ppy) - rf_rate) / (dstd * np.sqrt(ppy)) if dstd > 0 else 0.0

    cum = (1.0 + r).cumprod()
    dd = cum / cum.cummax() - 1.0
    max_dd = float(dd.min()) if len(dd) else 0.0
    calmar = float(cagr / abs(max_dd)) if max_dd < 0 else np.nan
    ulcer = float(np.sqrt((dd ** 2).mean())) if len(dd) else 0.0

    pos = float(r[r > 0].sum())
    neg = float(-r[r < 0].sum())
    profit_factor = pos / neg if neg > 0 else np.inf
    bar_hit = float((r[sig == 1] > 0).mean()) if (sig == 1).any() else np.nan

    excess = r - m
    te = float(excess.std()) * np.sqrt(ppy) if excess.std() > 0 else 0.0
    info_ratio = (float(excess.mean()) * ppy) / te if te > 0 else 0.0
    var_m = float(m.var())
    beta = float(r.cov(m) / var_m) if var_m > 0 else np.nan
    alpha_ann = float((r.mean() - beta * m.mean()) * ppy) if beta == beta else np.nan

    q95, q05 = float(r.quantile(0.95)), float(r.quantile(0.05))
    tail_ratio = q95 / abs(q05) if q05 != 0 else np.nan

    monthly = dfr["Net_Returns"].resample("ME").apply(lambda x: (1.0 + x).prod() - 1.0)
    monthly = monthly.dropna()
    pct_pos_months = float((monthly > 0).mean()) if len(monthly) else np.nan

    trade_stats = _round_trip_stats(dfr)
    n_flips = int((dfr["Signal_Change"] > 0).sum())

    metrics = {
        "Net Return": total_net,
        "Gross Return": total_gross,
        "CAGR": cagr,
        "Market Return": total_mkt,
        "Market CAGR": mkt_cagr,
        "Excess vs Market": total_net - total_mkt,
        "Ann. Volatility": vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "Calmar": calmar,
        "Max Drawdown": max_dd,
        "Max DD Bars": int(_max_dd_duration(dd.to_numpy())),
        "Ulcer Index": ulcer,
        "Profit Factor": profit_factor,
        "Bar Hit Rate": bar_hit,
        "Time in Market": float((sig == 1).mean()),
        "Turnover (flips)": n_flips,
        "Beta vs Market": beta,
        "Alpha (ann.)": alpha_ann,
        "Information Ratio": info_ratio,
        "Skew": float(r.skew()) if len(r) > 2 else np.nan,
        "Kurtosis": float(r.kurtosis()) if len(r) > 3 else np.nan,
        "Tail Ratio": tail_ratio,
        "Positive Months": pct_pos_months,
        "Bars / Year": ppy,
        "Years": years,
    }
    metrics.update(trade_stats)
    return metrics


def _classification_metrics(y_true, y_prob):
    from sklearn.metrics import (
        accuracy_score, balanced_accuracy_score, precision_score, recall_score,
        f1_score, roc_auc_score, log_loss,
    )
    y_true = np.asarray(y_true).reshape(-1)
    y_prob = np.asarray(y_prob).reshape(-1)
    y_hat = (y_prob >= 0.5).astype(int)
    try:
        auc = float(roc_auc_score(y_true, y_prob))
    except ValueError:
        auc = np.nan
    try:
        ll = float(log_loss(y_true, np.clip(y_prob, 1e-6, 1 - 1e-6)))
    except ValueError:
        ll = np.nan
    return {
        "Accuracy": float(accuracy_score(y_true, y_hat)),
        "Balanced Accuracy": float(balanced_accuracy_score(y_true, y_hat)),
        "Precision": float(precision_score(y_true, y_hat, zero_division=0)),
        "Recall": float(recall_score(y_true, y_hat, zero_division=0)),
        "F1": float(f1_score(y_true, y_hat, zero_division=0)),
        "ROC AUC": auc,
        "Log Loss": ll,
    }


def compare_models_backtest(
    models, val_data, df, tc_bps=5, slippage_bps=5, rf_rate=0.0, show=True
):
    """
    Backtest every candidate on the same cost-aware long/cash rule, print a
    comparison table, plot equity curves, and pick a winner by average rank
    across trading metrics.

    Parameters
    ----------
    models : dict[str, keras model | tuner | (model, history)]
    val_data : TimeseriesGenerator (or similar) used for predictions
    df : price frame with a 'close' column (e.g. df_boruta)
    """
    from IPython.display import display
    import matplotlib.dates as mdates
    from matplotlib.ticker import FuncFormatter

    if isinstance(val_data, tuple):
        y_true = np.asarray(val_data[1]).reshape(-1)
    else:
        y_true = np.concatenate(
            [(y.numpy() if hasattr(y, "numpy") else y) for _, y in val_data],
            axis=0,
        ).reshape(-1)

    rows = []
    equity = {}
    frames = {}
    model_objs = {}

    print("=" * 88)
    print("  MULTI-MODEL BACKTEST  |  lag-1 execution, "
          f"{tc_bps}+{slippage_bps} bps round-trip friction")
    print("=" * 88)

    for name, obj in models.items():
        model = _extract_keras_model(obj)
        model_objs[name] = model
        y_prob = model.predict(val_data, verbose=0)
        ypred = (np.asarray(y_prob).reshape(-1) >= 0.5).astype(int)
        if len(ypred) != len(y_true):
            n = min(len(ypred), len(y_true))
            ypred, y_prob, y_true_i = ypred[-n:], np.asarray(y_prob).reshape(-1)[-n:], y_true[-n:]
        else:
            y_true_i = y_true

        dfr = _align_cost_aware_backtest(df, ypred, tc_bps=tc_bps, slippage_bps=slippage_bps)
        frames[name] = dfr
        equity[name] = dfr["Cumulative_Strategy"]

        clf = _classification_metrics(y_true_i, y_prob)
        bt = _compute_strategy_metrics(dfr, rf_rate=rf_rate)
        row = {"Model": name, **clf, **bt}
        rows.append(row)

    mkt_name = "Buy & Hold"
    any_frame = next(iter(frames.values()))
    equity[mkt_name] = any_frame["Cumulative_Market"]
    mkt_metrics = _compute_strategy_metrics(
        any_frame.assign(
            Gross_Returns=any_frame["Market_Returns"].fillna(0.0),
            Net_Returns=any_frame["Market_Returns"].fillna(0.0),
            Executed_Signal=1.0,
            Signal_Change=0.0,
        ),
        rf_rate=rf_rate,
    )
    mkt_row = {"Model": mkt_name}
    for k in rows[0]:
        if k == "Model":
            continue
        mkt_row[k] = np.nan if k in {
            "Accuracy", "Balanced Accuracy", "Precision", "Recall", "F1", "ROC AUC", "Log Loss"
        } else mkt_metrics.get(k, np.nan)
    mkt_row["Net Return"] = mkt_metrics["Net Return"]
    mkt_row["Gross Return"] = mkt_metrics["Gross Return"]
    mkt_row["CAGR"] = mkt_metrics["CAGR"]
    mkt_row["Time in Market"] = 1.0
    mkt_row["Round Trips"] = 0
    mkt_row["Turnover (flips)"] = 0
    rows.append(mkt_row)

    metrics_df = pd.DataFrame(rows).set_index("Model")

    # Max Drawdown is signed (negative). Higher (closer to 0) is better, so it
    # belongs with rank_higher — ranking it as "lower is better" would reward
    # the worst drawdown.
    rank_higher = [
        "Net Return", "CAGR", "Sharpe", "Sortino", "Calmar",
        "Profit Factor", "Excess vs Market", "Trade Win Rate", "Information Ratio",
        "Max Drawdown",
    ]
    rank_lower = ["Ulcer Index"]
    cand = [ix for ix in metrics_df.index if ix != mkt_name]
    rank_src = metrics_df.loc[cand].replace([np.inf], 1e6).replace([-np.inf], -1e6)
    rank_df = pd.DataFrame(index=cand)
    for col in rank_higher:
        if col in rank_src:
            rank_df[col] = rank_src[col].rank(ascending=False, method="min")
    for col in rank_lower:
        if col in rank_src:
            rank_df[col] = rank_src[col].rank(ascending=True, method="min")
    rank_df["Avg Rank"] = rank_df.mean(axis=1)
    rank_df["_sharpe"] = metrics_df.loc[cand, "Sharpe"]
    rank_df = rank_df.sort_values(["Avg Rank", "_sharpe"], ascending=[True, False])
    rank_df = rank_df.drop(columns=["_sharpe"])
    winner_name = rank_df.index[0]
    winner_model = model_objs[winner_name]

    if show:
        ppy = metrics_df.loc[cand[0], "Bars / Year"]
        print(f"Annualization: {ppy:.0f} bars/year "
              f"(median bars/day x 252). Ranked on: {', '.join(rank_higher + rank_lower)}.")
        print()

        show_cols = [
            "Log Loss", "Accuracy", "Balanced Accuracy", "ROC AUC", "F1",
            "Net Return", "CAGR", "Excess vs Market", "Sharpe", "Sortino", "Calmar",
            "Max Drawdown", "Ulcer Index", "Profit Factor", "Time in Market",
            "Round Trips", "Trade Win Rate", "Avg Trade", "Payoff Ratio",
            "Information Ratio", "Positive Months",
        ]
        show_cols = [c for c in show_cols if c in metrics_df.columns]
        # Transpose: metrics as rows, models as columns (values unchanged)
        view_t = metrics_df[show_cols].T

        pct_rows = {
            "Accuracy", "Balanced Accuracy", "Net Return", "CAGR", "Excess vs Market",
            "Max Drawdown", "Time in Market", "Trade Win Rate", "Bar Hit Rate",
            "Positive Months", "Avg Trade",
        }
        int_rows = {"Round Trips", "Turnover (flips)", "Max DD Bars"}
        ratio_rows = {
            "Log Loss", "ROC AUC", "F1", "Precision", "Recall",
            "Sharpe", "Sortino", "Calmar", "Ulcer Index", "Profit Factor",
            "Payoff Ratio", "Information Ratio", "Beta vs Market", "Tail Ratio",
        }
        hi_max_rows = [
            r for r in [
                "Accuracy", "Balanced Accuracy", "ROC AUC", "F1", "Net Return", "CAGR",
                "Excess vs Market", "Sharpe", "Sortino", "Calmar", "Profit Factor",
                "Trade Win Rate", "Information Ratio", "Positive Months", "Max Drawdown",
            ] if r in view_t.index
        ]
        hi_min_rows = [r for r in ["Log Loss", "Ulcer Index"] if r in view_t.index]

        styled = view_t.style
        for metric in view_t.index:
            loc = pd.IndexSlice[metric, :]
            if metric in pct_rows:
                styled = styled.format("{:.2%}", subset=loc, na_rep="—")
            elif metric in int_rows:
                styled = styled.format("{:.0f}", subset=loc, na_rep="—")
            elif metric in ratio_rows:
                styled = styled.format("{:.3f}", subset=loc, na_rep="—")
            else:
                styled = styled.format("{:.3f}", subset=loc, na_rep="—")
        if hi_max_rows:
            styled = styled.highlight_max(subset=pd.IndexSlice[hi_max_rows, :], axis=1, color="#c6efce")
        if hi_min_rows:
            styled = styled.highlight_min(subset=pd.IndexSlice[hi_min_rows, :], axis=1, color="#c6efce")
        display(styled)

        print("\nAverage rank (1 = best). Winner = lowest Avg Rank.")
        rank_t = rank_df.T
        rank_styled = rank_t.style.format("{:.2f}")
        rank_styled = rank_styled.highlight_min(axis=1, color="#c6efce")
        display(rank_styled)

        print("\n" + "=" * 88)
        print(f"  BACKTEST WINNER: {winner_name}  (Avg Rank {rank_df.loc[winner_name, 'Avg Rank']:.2f})")
        print("=" * 88)

        colors = {
            "Baseline Model": "#1f77b4",
            "Random Search": "#ff7f0e",
            "Bayesian Optimization": "#2ca02c",
            "Hyperband": "#d62728",
            mkt_name: "#7f7f7f",
        }
        fig, axes = plt.subplots(2, 1, figsize=(16, 11), gridspec_kw={"height_ratios": [2.2, 1]})
        ax = axes[0]
        for name, series in equity.items():
            lw = 2.6 if name == winner_name else (1.5 if name == mkt_name else 1.8)
            ls = "--" if name == mkt_name else "-"
            ax.plot(series.index, series.values, label=name, color=colors.get(name),
                    linewidth=lw, linestyle=ls, alpha=0.95 if name == winner_name else 0.8)
        ax.set_title("Net Equity Curves (cost-aware long/cash vs Buy & Hold)", fontsize=15, fontweight="bold")
        ax.set_ylabel("Growth of 1.0")
        ax.grid(True, linestyle="--", alpha=0.4)
        ax.legend(loc="upper left", fontsize=10)
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

        axb = axes[1]
        bar_metrics = ["Sharpe", "Sortino", "Calmar"]
        x = np.arange(len(cand))
        width = 0.25
        for i, met in enumerate(bar_metrics):
            vals = metrics_df.loc[cand, met].replace([np.inf, -np.inf], np.nan).fillna(0.0)
            axb.bar(x + i * width, vals.values, width, label=met)
        axb.set_xticks(x + width)
        axb.set_xticklabels(cand, rotation=15, ha="right")
        axb.axhline(0, color="black", linewidth=1)
        axb.set_ylabel("Ratio")
        axb.set_title("Risk-Adjusted Ratios", fontsize=13, fontweight="bold")
        axb.legend()
        axb.grid(axis="y", linestyle="--", alpha=0.4)
        axb.spines["top"].set_visible(False)
        axb.spines["right"].set_visible(False)
        plt.tight_layout()
        plt.show()

        fig2, ax2 = plt.subplots(figsize=(16, 4.5))
        dd_colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
        for i, name in enumerate(cand):
            dfr = frames[name]
            cum = (1.0 + dfr["Net_Returns"]).cumprod()
            dd = cum / cum.cummax() - 1.0
            ax2.plot(dd.index, dd.values, label=name, color=dd_colors[i % len(dd_colors)], linewidth=1.6)
        ax2.set_title("Drawdown (net of costs)", fontsize=13, fontweight="bold")
        ax2.set_ylabel("Drawdown")
        ax2.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y:.1%}"))
        ax2.grid(True, linestyle="--", alpha=0.4)
        ax2.legend(loc="lower left")
        plt.tight_layout()
        plt.show()

    return {
        "metrics": metrics_df,
        "ranks": rank_df,
        "equity_curves": equity,
        "frames": frames,
        "winner_name": winner_name,
        "winner_model": winner_model,
    }


### MCDM model selection (TOPSIS + weighted composite)
- Input: `bt_results['metrics']` from `compare_models_backtest` (models as rows)
- Maximizes return, minimizes risk, maximizes stability via category weights
- Call `model_selector(...)` after the backtest comparison


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Literal, Mapping, Optional, Sequence, Tuple, Union

Direction = Literal["+", "-"]
Category = Literal["return", "risk", "stability"]
SelectMethod = Literal["topsis", "composite", "consensus"]


@dataclass(frozen=True)
class MetricSpec:
    """One decision criterion: name in the metrics table, direction, category."""

    name: str
    direction: Direction  # '+' maximize, '-' minimize
    category: Category


@dataclass(frozen=True)
class SelectorWeights:
    """Category weights. They are normalized to sum to 1 inside model_selector()."""

    return_weight: float = 1.0 / 3.0
    risk_weight: float = 1.0 / 3.0
    stability_weight: float = 1.0 / 3.0

    def normalized(self) -> Dict[Category, float]:
        total = self.return_weight + self.risk_weight + self.stability_weight
        if total <= 0:
            raise ValueError("Category weights must sum to a positive number.")
        if min(self.return_weight, self.risk_weight, self.stability_weight) < 0:
            raise ValueError("Category weights must be non-negative.")
        return {
            "return": self.return_weight / total,
            "risk": self.risk_weight / total,
            "stability": self.stability_weight / total,
        }


DEFAULT_METRIC_SPECS: Tuple[MetricSpec, ...] = (
    MetricSpec("Net Return", "+", "return"),
    MetricSpec("CAGR", "+", "return"),
    MetricSpec("Profit Factor", "+", "return"),
    MetricSpec("Max Drawdown", "-", "risk"),
    MetricSpec("Ulcer Index", "-", "risk"),
    MetricSpec("Sharpe", "+", "stability"),
    MetricSpec("Sortino", "+", "stability"),
    MetricSpec("Calmar", "+", "stability"),
    MetricSpec("Positive Months", "+", "stability"),
)


@dataclass
class ModelSelectionResult:
    """Container returned by model_selector()."""

    winner_name: str
    winner_model: object
    method: SelectMethod
    weights: SelectorWeights
    metric_weights: pd.Series
    decision_matrix: pd.DataFrame
    normalized: pd.DataFrame
    topsis: pd.DataFrame
    composite: pd.DataFrame
    combined: pd.DataFrame
    category_scores: pd.DataFrame
    summary: str
    specs_used: Tuple[MetricSpec, ...]


def _as_metrics_frame(metrics: Union[pd.DataFrame, Mapping]) -> pd.DataFrame:
    if isinstance(metrics, pd.DataFrame):
        df = metrics.copy()
    elif isinstance(metrics, Mapping) and "metrics" in metrics:
        df = metrics["metrics"].copy()
    else:
        raise TypeError(
            "metrics must be a DataFrame (models x metrics) or the dict "
            "returned by compare_models_backtest()."
        )
    if df.empty:
        raise ValueError("Metrics table is empty.")
    return df


def _prepare_column(series: pd.Series, spec: MetricSpec) -> Tuple[pd.Series, Direction]:
    """
    Coerce a metric to a finite numeric series and a TOPSIS direction.

    Max Drawdown is stored as a signed negative number. Minimizing the raw
    value would prefer the worst drawdown, so it is converted to magnitude
    and treated as a cost ('-').
    """
    s = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
    direction = spec.direction
    if spec.name == "Max Drawdown":
        finite = s.dropna()
        if len(finite) and (finite <= 0).all():
            s = s.abs()
            direction = "-"
    if s.notna().sum() == 0:
        raise ValueError(f"Metric '{spec.name}' is missing or non-numeric for all models.")
    if direction == "+":
        s = s.fillna(s.min())
    else:
        s = s.fillna(s.max())
    return s.astype(float), direction


def _category_metric_weights(
    specs: Sequence[MetricSpec],
    cat_weights: Mapping[Category, float],
) -> pd.Series:
    """Split each category weight equally across its available metrics."""
    w = {}
    by_cat: Dict[Category, List[str]] = {"return": [], "risk": [], "stability": []}
    for spec in specs:
        by_cat[spec.category].append(spec.name)
    for cat, names in by_cat.items():
        if not names:
            continue
        share = cat_weights[cat] / len(names)
        for name in names:
            w[name] = share
    weights = pd.Series(w, dtype=float)
    total = float(weights.sum())
    if total <= 0:
        raise ValueError("All metric weights collapsed to zero.")
    return weights / total


def _topsis(
    decision: pd.DataFrame,
    directions: Mapping[str, Direction],
    weights: pd.Series,
) -> pd.DataFrame:
    """Classical TOPSIS. Returns S+, S-, closeness C* in [0, 1]."""
    X = decision.to_numpy(dtype=float)
    denom = np.sqrt((X ** 2).sum(axis=0))
    denom = np.where(denom == 0.0, 1.0, denom)
    R = X / denom
    w = weights.reindex(decision.columns).to_numpy(dtype=float)
    V = R * w

    ideal_best = np.empty(V.shape[1])
    ideal_worst = np.empty(V.shape[1])
    for j, col in enumerate(decision.columns):
        if directions[col] == "+":
            ideal_best[j] = np.nanmax(V[:, j])
            ideal_worst[j] = np.nanmin(V[:, j])
        else:
            ideal_best[j] = np.nanmin(V[:, j])
            ideal_worst[j] = np.nanmax(V[:, j])

    s_plus = np.sqrt(((V - ideal_best) ** 2).sum(axis=1))
    s_minus = np.sqrt(((V - ideal_worst) ** 2).sum(axis=1))
    closeness = s_minus / np.where((s_plus + s_minus) == 0.0, 1.0, s_plus + s_minus)

    out = pd.DataFrame(
        {
            "S+ (to ideal)": s_plus,
            "S- (to anti-ideal)": s_minus,
            "TOPSIS C*": closeness,
        },
        index=decision.index,
    )
    out["TOPSIS Rank"] = out["TOPSIS C*"].rank(ascending=False, method="min")
    return out.sort_values("TOPSIS C*", ascending=False)


def _minmax_benefit(series: pd.Series, direction: Direction) -> pd.Series:
    """Scale to [0, 1] where 1 is always better. Constant columns -> 0.5."""
    lo, hi = float(series.min()), float(series.max())
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return pd.Series(0.5, index=series.index)
    if direction == "+":
        return (series - lo) / (hi - lo)
    return (hi - series) / (hi - lo)


def _composite_index(
    decision: pd.DataFrame,
    specs: Sequence[MetricSpec],
    directions: Mapping[str, Direction],
    cat_weights: Mapping[Category, float],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Weighted composite index from min-max scores, plus category aggregates."""
    benefit = pd.DataFrame(index=decision.index)
    for spec in specs:
        benefit[spec.name] = _minmax_benefit(decision[spec.name], directions[spec.name])

    cat_scores = pd.DataFrame(index=decision.index)
    for cat in ("return", "risk", "stability"):
        cols = [s.name for s in specs if s.category == cat]
        cat_scores[cat.capitalize()] = benefit[cols].mean(axis=1) if cols else 0.5

    composite = (
        cat_weights["return"] * cat_scores["Return"]
        + cat_weights["risk"] * cat_scores["Risk"]
        + cat_weights["stability"] * cat_scores["Stability"]
    )
    cat_scores["Composite Index"] = composite
    cat_scores["Composite Rank"] = composite.rank(ascending=False, method="min")
    cat_scores = cat_scores.sort_values("Composite Index", ascending=False)
    return cat_scores, benefit


def _build_summary(
    winner: str,
    combined: pd.DataFrame,
    category_scores: pd.DataFrame,
    decision: pd.DataFrame,
    specs: Sequence[MetricSpec],
    directions: Mapping[str, Direction],
    cat_weights: Mapping[Category, float],
    method: SelectMethod,
) -> str:
    others = [i for i in combined.index if i != winner]
    lines = [
        f"Selected model: {winner}  (method = {method}).",
        (
            f"Category weights — Return {cat_weights['return']:.0%}, "
            f"Risk {cat_weights['risk']:.0%}, Stability {cat_weights['stability']:.0%}."
        ),
    ]
    wrow = combined.loc[winner]
    lines.append(
        f"{winner} TOPSIS C* = {wrow['TOPSIS C*']:.3f} (rank {int(wrow['TOPSIS Rank'])}), "
        f"Composite = {wrow['Composite Index']:.3f} (rank {int(wrow['Composite Rank'])}), "
        f"consensus rank = {int(wrow['Consensus Rank'])}."
    )
    cats = category_scores.loc[winner, ["Return", "Risk", "Stability"]]
    lines.append(
        f"Category scores (0–1, higher is better): "
        f"Return {cats['Return']:.3f}, Risk {cats['Risk']:.3f}, Stability {cats['Stability']:.3f}."
    )
    if others:
        runner = others[0]
        rrow = combined.loc[runner]
        lines.append(
            f"Runner-up: {runner} (TOPSIS C* {rrow['TOPSIS C*']:.3f}, "
            f"Composite {rrow['Composite Index']:.3f})."
        )
        gaps = []
        for spec in specs:
            a, b = decision.loc[winner, spec.name], decision.loc[runner, spec.name]
            better = (a > b) if directions[spec.name] == "+" else (a < b)
            if better:
                gaps.append(f"{spec.name} ({a:.4g} vs {b:.4g})")
        if gaps:
            lines.append(f"{winner} beats {runner} on: {', '.join(gaps)}.")
        else:
            lines.append(
                f"{winner} is preferred to {runner} on the weighted combination, "
                "not because it dominates every single metric."
            )
        if len(others) > 1:
            rest = ", ".join(others[1:])
            lines.append(f"Remaining candidates in consensus order after {runner}: {rest}.")
    return "\n".join(lines)


def _plot_selection(
    combined: pd.DataFrame,
    category_scores: pd.DataFrame,
    winner: str,
) -> None:
    names = list(combined.index)
    x = np.arange(len(names))
    width = 0.35
    fig = plt.figure(figsize=(16, 5.4))

    ax = fig.add_subplot(1, 2, 1)
    ax.bar(x - width / 2, combined["TOPSIS C*"].values, width, label="TOPSIS C*", color="#1f77b4")
    ax.bar(x + width / 2, combined["Composite Index"].values, width, label="Composite Index", color="#ff7f0e")
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=15, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score (higher is better)")
    ax.set_title("MCDM Scores", fontweight="bold")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for i, name in enumerate(names):
        if name == winner:
            ax.axvline(i, color="#2ca02c", alpha=0.15, linewidth=24)

    axr = fig.add_subplot(1, 2, 2, polar=True)
    cats = ["Return", "Risk", "Stability"]
    angles = np.linspace(0, 2 * np.pi, len(cats), endpoint=False)
    angles = np.concatenate([angles, angles[:1]])
    palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
    for i, name in enumerate(names):
        vals = category_scores.loc[name, cats].to_numpy(dtype=float)
        vals = np.concatenate([vals, vals[:1]])
        lw = 2.6 if name == winner else 1.4
        axr.plot(angles, vals, label=name, color=palette[i % len(palette)], linewidth=lw)
        axr.fill(angles, vals, color=palette[i % len(palette)], alpha=0.08 if name != winner else 0.18)
    axr.set_xticks(angles[:-1])
    axr.set_xticklabels(cats)
    axr.set_ylim(0, 1.0)
    axr.set_title("Category scores (benefit-scaled)", fontweight="bold", pad=16)
    axr.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=9)

    plt.tight_layout()
    plt.show()


def model_selector(
    metrics: Union[pd.DataFrame, Mapping],
    *,
    models: Optional[Mapping[str, object]] = None,
    weights: Optional[SelectorWeights] = None,
    method: SelectMethod = "consensus",
    exclude: Sequence[str] = ("Buy & Hold",),
    specs: Optional[Sequence[MetricSpec]] = None,
    show: bool = True,
) -> ModelSelectionResult:
    """
    Select an optimal model from a backtest metrics table using TOPSIS and
    a weighted composite index.

    Parameters
    ----------
    metrics
        DataFrame with models as the index and metric names as columns
        (``bt_results['metrics']`` from ``compare_models_backtest``), or the
        full ``bt_results`` dict.
    models
        Optional ``{name: keras model}`` mapping used to return ``winner_model``.
        Typically the ``models`` dict built in ``2_model_building.ipynb``.
    weights
        ``SelectorWeights(return_weight, risk_weight, stability_weight)``.
        Defaults to equal category weights.
    method
        ``'topsis'``, ``'composite'``, or ``'consensus'`` (average of the two
        ranks; TOPSIS C* breaks ties). Default ``'consensus'``.
    exclude
        Index labels to drop before scoring (benchmark rows such as Buy & Hold).
    specs
        Optional override of ``DEFAULT_METRIC_SPECS``.
    show
        If True, display ranking tables, charts, and the text summary.

    Returns
    -------
    ModelSelectionResult

    Notes
    -----
    Return criteria (maximize): Net Return, CAGR, Profit Factor.
    Risk criteria (minimize): Max Drawdown magnitude, Ulcer Index.
    Stability criteria (maximize): Sharpe, Sortino, Calmar, Positive Months.

    Example
    -------
    >>> sel = model_selector(
    ...     bt_results["metrics"],
    ...     models=models,
    ...     weights=SelectorWeights(0.40, 0.30, 0.30),
    ...     method="consensus",
    ... )
    >>> sel.winner_name
    """
    from IPython.display import display

    frame = _as_metrics_frame(metrics)
    drop = [x for x in exclude if x in frame.index]
    frame = frame.drop(index=drop, errors="ignore")
    if frame.empty:
        raise ValueError("No candidate models remain after applying `exclude`.")

    weights = weights or SelectorWeights()
    cat_w = weights.normalized()
    specs = tuple(specs) if specs is not None else DEFAULT_METRIC_SPECS

    used: List[MetricSpec] = []
    directions: Dict[str, Direction] = {}
    cols = {}
    skipped = []
    for spec in specs:
        if spec.name not in frame.columns:
            skipped.append(spec.name)
            continue
        try:
            series, direction = _prepare_column(frame[spec.name], spec)
        except ValueError:
            skipped.append(spec.name)
            continue
        cols[spec.name] = series
        directions[spec.name] = direction
        used.append(MetricSpec(spec.name, direction, spec.category))

    if len(used) < 2:
        raise ValueError(
            "Need at least two usable metrics. Missing/invalid: "
            + (", ".join(skipped) if skipped else "none")
        )
    if skipped and show:
        print("Skipping unavailable metrics:", ", ".join(skipped))

    decision = pd.DataFrame(cols, index=frame.index)
    metric_w = _category_metric_weights(used, cat_w)
    topsis = _topsis(decision, directions, metric_w)
    category_scores, benefit = _composite_index(decision, used, directions, cat_w)

    combined = pd.DataFrame(index=decision.index)
    combined["TOPSIS C*"] = topsis["TOPSIS C*"]
    combined["TOPSIS Rank"] = topsis["TOPSIS Rank"]
    combined["Composite Index"] = category_scores["Composite Index"]
    combined["Composite Rank"] = category_scores["Composite Rank"]
    combined["Return"] = category_scores["Return"]
    combined["Risk"] = category_scores["Risk"]
    combined["Stability"] = category_scores["Stability"]
    combined["Avg Rank"] = (combined["TOPSIS Rank"] + combined["Composite Rank"]) / 2.0
    combined["Consensus Rank"] = combined["Avg Rank"].rank(method="min")
    # Tie-break: better TOPSIS C*
    combined = combined.sort_values(
        ["Consensus Rank", "TOPSIS C*"], ascending=[True, False]
    )

    if method == "topsis":
        winner = str(topsis.index[0])
    elif method == "composite":
        winner = str(category_scores.index[0])
    else:
        winner = str(combined.index[0])

    winner_model = None
    if models is not None and winner in models:
        try:
            winner_model = _extract_keras_model(models[winner])
        except Exception:
            winner_model = models[winner]

    summary = _build_summary(
        winner, combined, category_scores, decision, used, directions, cat_w, method
    )

    if show:
        print("=" * 88)
        print("  MODEL SELECTOR  |  TOPSIS + Weighted Composite Index")
        print("=" * 88)
        print(
            f"Weights (normalized): Return {cat_w['return']:.0%} | "
            f"Risk {cat_w['risk']:.0%} | Stability {cat_w['stability']:.0%}"
        )
        print("Per-metric TOPSIS weights (category split equally inside each bucket):")
        wtab = pd.DataFrame(
            {
                "Weight": metric_w,
                "Direction": pd.Series(directions),
                "Category": {s.name: s.category for s in used},
            }
        )
        display(wtab.T)

        print("\nDecision matrix (risk metrics as magnitudes to minimize):")
        display(decision.T.style.format("{:.4f}"))

        print("\nTOPSIS (C* closer to 1 is better):")
        display(
            topsis.T.style.format("{:.4f}").highlight_max(
                subset=pd.IndexSlice[["TOPSIS C*"], :], axis=1, color="#c6efce"
            )
        )

        print("\nWeighted composite index (category scores 0–1):")
        display(
            category_scores.T.style.format("{:.4f}").highlight_max(
                subset=pd.IndexSlice[["Return", "Risk", "Stability", "Composite Index"], :],
                axis=1,
                color="#c6efce",
            )
        )

        print("\nConsensus ranking (average of TOPSIS rank and Composite rank):")
        show_cols = [
            "TOPSIS C*", "TOPSIS Rank", "Composite Index", "Composite Rank",
            "Avg Rank", "Consensus Rank", "Return", "Risk", "Stability",
        ]
        display(
            combined[show_cols].T.style.format("{:.4f}").highlight_min(
                subset=pd.IndexSlice[["TOPSIS Rank", "Composite Rank", "Avg Rank", "Consensus Rank"], :],
                axis=1,
                color="#c6efce",
            )
        )

        print("\n" + summary)
        print("\n" + "=" * 88)
        print(f"  MCDM WINNER: {winner}  (method={method})")
        print("=" * 88)
        _plot_selection(combined, category_scores, winner)

    return ModelSelectionResult(
        winner_name=winner,
        winner_model=winner_model,
        method=method,
        weights=weights,
        metric_weights=metric_w,
        decision_matrix=decision,
        normalized=benefit,
        topsis=topsis,
        composite=category_scores,
        combined=combined,
        category_scores=category_scores,
        summary=summary,
        specs_used=tuple(used),
    )
